# Planificacion automatica aplicada al Senku
## Experimentacion - Convocatoria de junio

Notebook que acompana al sistema desarrollado en `senku/src`. Aqui se realizan los experimentos sobre las cinco variantes del Senku (Figura 3 de la propuesta), comparando:

1. **BFS** sobre la representacion interna (parte comun, sin heuristica).
2. **Beam Search** con la heuristica pagoda (algoritmo especifico de la convocatoria de junio).
3. **Beam Search** con la heuristica compuesta (pagoda + aislamiento + compacidad) como ampliacion.

Para cada variante se mide tiempo de ejecucion, nodos expandidos y exito.
Las variantes 1, 3 y 5 son las requeridas por el enunciado; las variantes 2 y 4 se incluyen para comparar el comportamiento en problemas mas pequenos.

In [ ]:
import sys
from pathlib import Path
# Permite ejecutar el notebook desde senku/notebooks/
RAIZ = Path.cwd().parent.parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from senku.src.tableros import TABLEROS, dibuja_tablero, VARIANTES_OBLIGATORIAS
from senku.src.estado import ProblemaSenku
from senku.src.heuristicas import (
    pagoda_clasica, pagoda_uniforme,
    heuristica_pagoda, heuristica_compuesta
)
from senku.src.busqueda import (
    busqueda_primero_anchura,
    beam_search,
    beam_search_con_reinicios,
)

print(f'Variantes definidas: {list(TABLEROS.keys())}')
print(f'Obligatorias (junio): {VARIANTES_OBLIGATORIAS}')

## 1. Inspeccion de los tableros

Visualizamos los cinco tableros con su estado inicial (las casillas ocupadas se marcan con `o` y el hueco inicial con `.`).

In [ ]:
for numero, tablero in TABLEROS.items():
    obligatoria = ' (obligatoria)' if numero in VARIANTES_OBLIGATORIAS else ''
    print(f'\n=== Variante {numero}{obligatoria}: {tablero.nombre} ({len(tablero.casillas)} casillas) ===')
    print(dibuja_tablero(tablero, tablero.inicial_ocupadas))

## 2. Pagoda de los estados iniciales

Comprobamos que la asignacion clasica de pagoda cumple la cota a + b >= c en todas las ternas y calculamos la pagoda inicial y meta.

In [ ]:
def valida_pagoda(pesos, problema):
    """Verifica la condicion a + b >= c en todas las ternas de salto."""
    violaciones = []
    for d, s, h in problema.saltos:
        if pesos[d] + pesos[s] < pesos[h]:
            violaciones.append((d, s, h))
    return violaciones

from senku.src.heuristicas import valor_pagoda
for numero, tablero in TABLEROS.items():
    p = ProblemaSenku.desde_tablero(tablero)
    pesos = pagoda_clasica(tablero)
    viol = valida_pagoda(pesos, p)
    pag_ini = valor_pagoda(p.inicial, pesos)
    pag_meta = sum(pesos[c] for c in p.meta_ocupadas if c in pesos)
    print(f'Var {numero}: violaciones={len(viol)} | pagoda inicial={pag_ini} | pagoda meta={pag_meta} | exceso={pag_ini - pag_meta}')

## 3. Experimento principal: comparacion de algoritmos

Para cada variante ejecutamos BFS (acotada en numero de nodos) y Beam Search con dos heuristicas distintas. Los resultados se acumulan en un DataFrame para visualizacion posterior.

In [ ]:
import pandas as pd

filas = []
LIMITE_NODOS_BFS = 50_000  # cota para que BFS no se eternice
BETA = 200
INTENTOS = 5
ITER_MAX = 60

for numero, tablero in TABLEROS.items():
    p = ProblemaSenku.desde_tablero(tablero)
    pesos = pagoda_clasica(tablero)
    h_pag = heuristica_pagoda(p, pesos)
    h_com = heuristica_compuesta(p, pesos)

    r = busqueda_primero_anchura(p, limite_nodos=LIMITE_NODOS_BFS)
    filas.append({
        'variante': numero, 'algoritmo': 'BFS', 'heuristica': '-',
        'exito': r.exito, 'movimientos': len(r.movimientos),
        'nodos': r.nodos_expandidos, 'tiempo_s': round(r.tiempo_segundos, 3),
    })

    r = beam_search_con_reinicios(p, h_pag, beta=BETA, intentos=INTENTOS, iteraciones_maximas=ITER_MAX)
    filas.append({
        'variante': numero, 'algoritmo': 'Beam', 'heuristica': 'pagoda',
        'exito': r.exito, 'movimientos': len(r.movimientos),
        'nodos': r.nodos_expandidos, 'tiempo_s': round(r.tiempo_segundos, 3),
    })

    r = beam_search_con_reinicios(p, h_com, beta=BETA, intentos=INTENTOS, iteraciones_maximas=ITER_MAX)
    filas.append({
        'variante': numero, 'algoritmo': 'Beam', 'heuristica': 'compuesta',
        'exito': r.exito, 'movimientos': len(r.movimientos),
        'nodos': r.nodos_expandidos, 'tiempo_s': round(r.tiempo_segundos, 3),
    })

df = pd.DataFrame(filas)
df

## 4. Influencia del parametro beta

Beam Search es incompleto y su capacidad de encontrar solucion depende mucho de la anchura del haz. Este experimento mide el efecto de beta sobre la profundidad alcanzada y el numero de exitos en multiples reinicios.

In [ ]:
VARIANTE_OBJETIVO = 1  # cruz inglesa
BETAS = [50, 100, 200, 500, 1000, 2000]
INTENTOS_BARRIDO = 5

p = ProblemaSenku.desde_tablero(TABLEROS[VARIANTE_OBJETIVO])
pesos = pagoda_clasica(p.tablero)
h = heuristica_compuesta(p, pesos)

filas_beta = []
for beta in BETAS:
    r = beam_search_con_reinicios(p, h, beta=beta, intentos=INTENTOS_BARRIDO, iteraciones_maximas=60)
    filas_beta.append({
        'beta': beta, 'exito': r.exito, 'mov': len(r.movimientos),
        'nodos': r.nodos_expandidos, 'tiempo_s': round(r.tiempo_segundos, 3)
    })
df_beta = pd.DataFrame(filas_beta)
df_beta

## 5. Modo relajado

El enunciado menciona que existen versiones mas relajadas del problema en las que basta con terminar con una unica pieza, en cualquier posicion. Repetimos el experimento bajo este criterio.

In [ ]:
filas_relax = []
for numero, tablero in TABLEROS.items():
    p = ProblemaSenku.desde_tablero(tablero, modo_relajado=True)
    pesos = pagoda_clasica(tablero)
    h = heuristica_compuesta(p, pesos)
    r = beam_search_con_reinicios(p, h, beta=500, intentos=5, iteraciones_maximas=60)
    filas_relax.append({
        'variante': numero, 'casillas': len(tablero.casillas),
        'exito': r.exito, 'mov': len(r.movimientos),
        'nodos': r.nodos_expandidos, 'tiempo_s': round(r.tiempo_segundos, 3)
    })
pd.DataFrame(filas_relax)

## 6. Lectura del sistema desde PDDL

Validacion del requisito de la convocatoria: el sistema recibe dos ficheros .pddl (dominio + problema) y los resuelve mediante beam search.

In [ ]:
from senku.src.lector_pddl import carga_problema_pddl

ruta_dominio = RAIZ / 'senku' / 'pddl' / 'dominio_senku.pddl'
ruta_problema = RAIZ / 'senku' / 'pddl' / 'problemas' / 'variante_2.pddl'

p = carga_problema_pddl(ruta_dominio, ruta_problema)
print(f'Cargado: {p.tablero.nombre}')
print(f'  Casillas: {len(p.tablero.casillas)}')
print(f'  Piezas iniciales: {len(p.inicial)}')
print(f'  Saltos definidos: {len(p.saltos)}')
print(f'  Meta ocupadas: {sorted(p.meta_ocupadas)}')
print()
print(dibuja_tablero(p.tablero, p.inicial))

## 7. Conclusiones experimentales

Los resultados confirman dos observaciones clave que se desarrollan en la memoria:

1. **BFS no escala**: el espacio de estados crece exponencialmente y BFS no encuentra solucion en las variantes medianas con un presupuesto razonable de nodos.
2. **Beam Search es sensible a la heuristica**: la pagoda admisible no contiene suficiente senal para evitar callejones sin salida; al combinarla con un termino que penaliza piezas aisladas y otro que favorece la compacidad, la profundidad alcanzada aumenta.
3. **Beta y reinicios son criticos**: ampliar el haz y multiplicar los reinicios estocasticos amplia significativamente la profundidad media alcanzada y, por tanto, las probabilidades de cerrar una solucion.